# SAM2 + transparent-background 背景除去パイプライン (Google Colab + Gradio 5)

このノートブックは **環境セットアップと起動** に専念し、実装本体は [`gradio_app_sam2_transparent_BG.py`](./gradio_app_sam2_transparent_BG.py) に集約しています（DRY 原則）。

**手順:**
1. **Cell 1**: 依存パッケージをインストール
2. **Cell 2**: Google Drive マウント & `PROJECT_ROOT` 解決 & チェックポイント配置
3. **Cell 3**: `gradio_app_sam2_transparent_BG.py` をサブプロセス起動

> **注意**:
> - Google Colab の GPU ランタイム (`ランタイム → ランタイムのタイプを変更 → T4 GPU`) で実行してください。
> - 停止はセルの停止ボタン（⏹）を押すとプロセスごと kill され GPU メモリも解放されます。
> - 実装の修正は `.py` 側で行い、ノートブックは触らないでください（一度の修正で済むようになります）。


In [ ]:
# ============================================================
# Cell 1: 依存関係インストール
# ============================================================
# Gradio 5 系（-q を外しておくとエラーが隠れない）
!pip install gradio==5.9.1

# 背景除去モデル
!pip install transparent-background

# Color decontamination 後処理
!pip install pymatting

# SAM2 (Meta 公式リポジトリ)
!pip install git+https://github.com/facebookresearch/sam2.git

# 画像処理基盤
!pip install opencv-python-headless pillow numpy

print("✅ Install done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.4/320.4 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.2/170.2 kB 16.6 MB/s eta 0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 15.0.1
    Uninstalling websockets-15.0.1:
      Successfully uninstalled websockets-15.0.1
  Attempting uninstall: markupsafe
    Found existing installation: MarkupSafe 3.0.3
    Uninstalling MarkupSafe-3.0.3:
      Successfully uninstalled MarkupSafe-3.0.3
  Attempting uninstall: aiofiles
    Found existing installation: aiofiles 24.1.0
    Uninstalling aiofiles-24.1.0:
      Successfully uninstalled aiofiles-24.1.0
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing installation: gradio 5.5

In [ ]:
# ============================================================
# Cell 2: Google Drive マウント + プロジェクトパス設定 + チェックポイント配置
# ============================================================
import os
import sys
from pathlib import Path

# ----- Google Colab 判定 -----
IS_COLAB = 'google.colab' in sys.modules

# ----- Colab の場合は Google Drive をマウント -----
# プロジェクト本体が Google Drive 上にあるためマウントが必須。
# 既にマウント済みなら何もしない。
COLAB_DRIVE_PROJECT = Path('/content/drive/MyDrive/AI_picasso/Matting-Anything')
if IS_COLAB:
    drive_root = Path('/content/drive/MyDrive')
    if not drive_root.exists():
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive')
        print('✅ Google Drive mounted at /content/drive')
    else:
        print('ℹ Google Drive already mounted')

# ----- プロジェクトルートを決定 -----
# 優先順位:
#   1. 環境変数 PROJECT_ROOT
#   2. Colab かつ Drive 上の所定パスが存在する場合 → そのパス
#   3. それ以外（ローカル）: 現在の作業ディレクトリ
def _detect_project_root() -> Path:
    env_root = os.environ.get('PROJECT_ROOT')
    if env_root:
        return Path(env_root).resolve()
    if IS_COLAB and COLAB_DRIVE_PROJECT.exists():
        return COLAB_DRIVE_PROJECT
    return Path.cwd().resolve()


PROJECT_ROOT = _detect_project_root()
print(f'PROJECT_ROOT = {PROJECT_ROOT}')

# Colab で Drive プロジェクト配下を作業ディレクトリにしておく
if IS_COLAB and PROJECT_ROOT == COLAB_DRIVE_PROJECT:
    os.chdir(PROJECT_ROOT)
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

# ----- 標準ディレクトリ（すべてプロジェクト内） -----
CKPT_ROOT       = PROJECT_ROOT / 'checkpoints'
SAM2_CKPT_DIR   = CKPT_ROOT / 'SAM2'
TB_CKPT_DIR     = CKPT_ROOT / 'transparent_BG'
INPUT_DIR       = PROJECT_ROOT / 'inputs'
OUTPUT_DIR      = PROJECT_ROOT / 'outputs'
for d in (SAM2_CKPT_DIR, TB_CKPT_DIR, INPUT_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ----- SAM2 チェックポイント -----
SAM2_CKPT_PATH   = SAM2_CKPT_DIR / 'sam2.1_hiera_large.pt'
SAM2_CONFIG_NAME = 'configs/sam2.1/sam2.1_hiera_l.yaml'  # パッケージ内パス

# ----- transparent-background チェックポイント（モード別） -----
TB_CKPT_BASE        = TB_CKPT_DIR / 'ckpt_base.pth'
TB_CKPT_FAST        = TB_CKPT_DIR / 'ckpt_fast.pth'
TB_CKPT_BASE_NIGHT  = TB_CKPT_DIR / 'ckpt_base_nightly.pth'


def fetch_if_missing(path: Path, url: str) -> None:
    """ファイルがなければダウンロードする（初回のみ）。"""
    if path.exists():
        print(f'Found: {path.relative_to(PROJECT_ROOT)}')
        return
    print(f'Downloading: {path.name}')
    rc = os.system(f'wget -q -O "{path}" "{url}"')
    if rc != 0 or not path.exists():
        raise RuntimeError(f'Download failed: {url}')
    print(f'Saved: {path.relative_to(PROJECT_ROOT)}')


# SAM2.1 large チェックポイント（未取得なら DL）
fetch_if_missing(
    SAM2_CKPT_PATH,
    'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt',
)

# transparent-background は初回 Remover() 呼び出し時に自動DLされる。
# checkpoints/transparent_BG/ に手動コピーすれば 2 回目以降の起動が高速。

print(f'\nSAM2 checkpoints  ({SAM2_CKPT_DIR}):')
for p in sorted(SAM2_CKPT_DIR.iterdir()):
    print(f'  - {p.name}  ({p.stat().st_size / 1e6:.1f} MB)')
print(f'\ntb checkpoints    ({TB_CKPT_DIR}):')
for p in sorted(TB_CKPT_DIR.iterdir()):
    print(f'  - {p.name}  ({p.stat().st_size / 1e6:.1f} MB)')
print(f'\nINPUT_DIR  = {INPUT_DIR}')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')


Mounted at /content/drive
✅ Google Drive mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/AI_picasso/Matting-Anything
Found: checkpoints/SAM2/sam2.1_hiera_large.pt

SAM2 checkpoints  (/content/drive/MyDrive/AI_picasso/Matting-Anything/checkpoints/SAM2):
  - sam2.1_hiera_large.pt  (898.1 MB)

tb checkpoints    (/content/drive/MyDrive/AI_picasso/Matting-Anything/checkpoints/transparent_BG):

INPUT_DIR  = /content/drive/MyDrive/AI_picasso/Matting-Anything/inputs
OUTPUT_DIR = /content/drive/MyDrive/AI_picasso/Matting-Anything/outputs


In [ ]:
# ============================================================
# Cell 3: Gradio アプリ起動
# ============================================================
# 実装本体は gradio_app_sam2_transparent_BG.py に集約（DRY）。
# このセルからサブプロセスとして起動する。
#
# ・Colab では --share を付けて公開 URL を発行
# ・ローカルでは --share を外して http://127.0.0.1:7860 で起動
# ・停止するにはセルの停止ボタン（⏹）を押す → プロセスごと kill され GPU メモリも解放される

import os
import sys

# Cell 2 で定義した PROJECT_ROOT 配下にあるアプリを起動する
APP_PATH = PROJECT_ROOT / "gradio_app_sam2_transparent_BG.py"
assert APP_PATH.exists(), f"アプリが見つかりません: {APP_PATH}"

# サブプロセスに設定を引き継ぐため環境変数を明示
# （`.py` 側は os.environ.get("PROJECT_ROOT") などで参照する）
os.environ["PROJECT_ROOT"]     = str(PROJECT_ROOT)
os.environ["SAM2_CKPT_PATH"]   = str(SAM2_CKPT_PATH)
os.environ["SAM2_CONFIG_NAME"] = SAM2_CONFIG_NAME

# Colab 判定: Colab では share=True が必須（127.0.0.1 にアクセスできないため）
IS_COLAB = "google.colab" in sys.modules
SHARE_FLAG = "--share" if IS_COLAB else ""

# `!` コマンドは Jupyter の system shell に渡され引用符が保持される。
# Windows / 日本語パスでも sys.executable + ダブルクォートで安全。
!{sys.executable} "{APP_PATH}" {SHARE_FLAG}


/usr/local/lib/python3.12/dist-packages/transparent_background/gui.py:24: UserWarning: Failed to import flet. Ignore this message when you do not need GUI mode.
  warnings.warn('Failed to import flet. Ignore this message when you do not need GUI mode.')
[INFO] PROJECT_ROOT = /content/drive/MyDrive/AI_picasso/Matting-Anything
[INFO] Device: cuda
[INFO] SAM2 loaded
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://d49a8d60234554f78a.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-def